## Data Pipeline
A. rir & source-receiver position & metadata (sourcex/srcx_recx.mat)  ---export_all_rir_data.py---> exported mono RIRs (.npy) + metadata (.json) + visualizations (.png) ---diffwave.preprocess()---> mel spectrograms (.spec.npy) for model training.

B. mesh (empty_room_x.stl) ---minaf feature extraction (this notebook)---> sampled feature vectors

C. spectrograms + features ---feature duplication/interpolation (this notebook)---> final training inputs for DiffWave

### Raw RIR Data Structure
Each RIR for a source-receiver pair is stored in:

`empty_room_data/source_x/srcx_recx.mat`

Each `.mat` file contains source/receiver positions, impulse response data, sampling rate, and related metadata.

### End-to-End Steps
1. Inspect one example `.mat` file with `DATA_DIR/inspect_empty_room.py` to verify field names and tensor shapes.
Starting folder structure:
```
empty_room_data/
├── source_x/
│   ├── srcx_recx.mat
├── empty_room_x.stl
```
2. Run `DATA_DIR/export_all_rir_data.py` to process all `.mat` files, extract relevant data, and save mono RIRs, metadata, and visualizations. The output structure will be:
```
empty_room_data/
├──...
├──exported_data/
   ├── mono_rir_npy/           (1 .npy files)
   ├── visualizations/         (20 HTML files)
   ├── all_metadata.json       (complete metadata)
   └── dataset_summary.txt     (human-readable summary)
```
3. Load the exported metadata and mesh file in this notebook, compute geometry-aware features for all source-receiver pairs using Minaf, and save the feature vectors for later use in training.
4. Convert the exported mono RIRs to mel spectrograms using DiffWave's preprocessing pipeline, and save the spectrograms for model training. Output structure:
```
empty_room_data/
├──...
├──exported_data/
    ├── wav                      
    │   ├── sourcex_recx.wav
    │   ├── sourcex_recx.wav.spec.npy
    ├── ...
```
4. This notebook uses functions from `preprocessing/data_utils.py` (`extract_features`, `probe_environment_dict`) to compute geometry-aware features for source and receiver locations. This code is adapted from the MiNaf paper. The output feature vectors will be saved as .csv files for later use in training. 
5. 

TL;DR: Final split dataset structure after all preprocessing steps:
```empty_room_data/
├──...
├──exported_data/
    ├── splits/
    │   ├── train/
    │   │   ├── wav/
    │   │   ├── features/
    │   ├── val/
    │   ├── test/
```
Feed them into DiffWave using
```bash
python -m diffwave /path/to/model/dir DATA_DIR/exported_data/splits/train/wav \
	--global_conditioning \
	--global_conditioning_suffix .npy \
    --global_conditioning_dir DATA_DIR/exported_data/splits/train/features
```


In [10]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from data_utils import probe_environment_dict

# Project paths (notebook is in preprocessing/)
DATA_ROOT = Path.cwd().parent.parent
DATA_DIR = DATA_ROOT / "empty_room_data" / "exported_data"
METADATA_PATH = DATA_DIR / "all_metadata.json"

# Update this to your mesh file path
MESH_PATH = DATA_ROOT / "empty_room_data" / "empty_room1.stl"

print(f"Project root: {DATA_ROOT}")
print(f"Metadata path: {METADATA_PATH}")
print(f"Mesh path: {MESH_PATH}")

Project root: /Users/thejavanoob/Research
Metadata path: /Users/thejavanoob/Research/empty_room_data/exported_data/all_metadata.json
Mesh path: /Users/thejavanoob/Research/empty_room_data/empty_room1.stl


### Load Exported Metadata
Read `all_metadata.json`, then build unique source and receiver point dictionaries in the format expected by `probe_environment_dict` (`name -> np.array([x, y, z])`).

In [3]:
if not METADATA_PATH.exists():
    raise FileNotFoundError(f"Metadata file not found: {METADATA_PATH}, make sure to run export_all_rir_data.py first to generate the metadata.")

with open(METADATA_PATH, "r") as f:
    all_metadata = json.load(f)

print(f"Loaded {len(all_metadata)} source-receiver entries")

source_points = {}
receiver_points = {}

for row in all_metadata:
    s_idx = int(row["source_idx"])
    r_idx = int(row["receiver_idx"])
    s_name = f"source_{s_idx:02d}"
    r_name = f"source_{s_idx:02d}_rec_{r_idx:02d}"

    source_points[s_name] = np.asarray(row["source_position"], dtype=np.float64)
    receiver_points[r_name] = np.asarray(row["receiver_position"], dtype=np.float64)

print(f"Unique source points: {len(source_points)}")
print(f"Receiver points: {len(receiver_points)}")
print("Example source:", next(iter(source_points.items())))
print("Example receiver:", next(iter(receiver_points.items())))

Loaded 1000 source-receiver entries
Unique source points: 20
Receiver points: 1000
Example source: ('source_00', array([2.636 , 2.4085, 1.1923]))
Example receiver: ('source_00_rec_00', array([5.0259, 2.8473, 1.9533]))


In [4]:
# Validate metadata schema and basic consistency checks
required_keys = {
    "source_idx", "receiver_idx", "source_position", "receiver_position",
    "distance", "sample_rate", "ir_length_samples", "ir_duration_seconds", "npy_file"
}

missing = []
bad_positions = 0
bad_distance = 0

for i, row in enumerate(all_metadata):
    row_keys = set(row.keys())
    if not required_keys.issubset(row_keys):
        missing.append((i, sorted(list(required_keys - row_keys))))

    src = np.asarray(row["source_position"], dtype=np.float64)
    rec = np.asarray(row["receiver_position"], dtype=np.float64)
    if src.shape != (3,) or rec.shape != (3,):
        bad_positions += 1
        continue

    calc_dist = float(np.linalg.norm(src - rec))
    if not np.isclose(calc_dist, float(row["distance"]), atol=1e-5):
        bad_distance += 1

print("Rows:", len(all_metadata))
print("Missing-key rows:", len(missing))
print("Bad position rows:", bad_positions)
print("Distance mismatch rows:", bad_distance)
if missing:
    print("First missing-key example:", missing[0])

Rows: 1000
Missing-key rows: 0
Bad position rows: 0
Distance mismatch rows: 0


### Convert Exported RIRs to WAV
Iterate through exported mono RIR NumPy files and convert them to mono 16-bit `.wav` files at the target sample rate.

In [ ]:
from pathlib import Path
from scipy.io import wavfile
from scipy.signal import resample_poly
import math
import sys

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from diffwave.params import params as diffwave_params

# Convert exported mono RIR .npy files to 16-bit mono .wav files.
MONO_RIR_DIR = DATA_DIR / "mono_rir_npy"
WAV_DIR = DATA_DIR / "wav"
TARGET_SAMPLE_RATE = int(diffwave_params.sample_rate)

if not MONO_RIR_DIR.exists():
    raise FileNotFoundError(
        f"RIR npy directory not found: {MONO_RIR_DIR}. Run export_all_rir_data.py first."
    )

WAV_DIR.mkdir(parents=True, exist_ok=True)

converted = 0
skipped = 0
for row in all_metadata:
    npy_name = row.get("npy_file")
    if not npy_name:
        skipped += 1
        continue

    npy_path = MONO_RIR_DIR / npy_name
    if not npy_path.exists():
        skipped += 1
        continue

    sr_in = int(row.get("sample_rate", TARGET_SAMPLE_RATE))
    rir = np.load(npy_path)
    rir = np.asarray(rir, dtype=np.float32).squeeze()

    if rir.ndim != 1:
        raise ValueError(f"Expected mono 1D IR in {npy_path}, got shape {rir.shape}")

    if sr_in != TARGET_SAMPLE_RATE:
        g = math.gcd(sr_in, TARGET_SAMPLE_RATE)
        up = TARGET_SAMPLE_RATE // g
        down = sr_in // g
        rir = resample_poly(rir, up, down).astype(np.float32)

    peak = float(np.max(np.abs(rir)))
    if peak > 1.0:
        rir = rir / peak

    wav_name = Path(npy_name).stem.replace("_mono", "") + ".wav"
    wav_path = WAV_DIR / wav_name
    wav_i16 = np.int16(np.clip(rir, -1.0, 1.0) * 32767.0)
    wavfile.write(wav_path, TARGET_SAMPLE_RATE, wav_i16)
    converted += 1

print(f"Converted {converted} RIRs to wav in {WAV_DIR}")
if skipped:
    print(f"Skipped {skipped} rows (missing npy_file field or missing npy path)")

Converted 1000 RIRs to wav in /Users/thejavanoob/Research/empty_room_data/exported_data/wav
Running diffwave.preprocess...


Preprocessing: 100%|██████████| 1000/1000 [00:18<00:00, 55.25it/s]


DiffWave spectrogram preprocessing complete.


### Extract Minaf Features
Run geometry-aware feature extraction for source and receiver points using `probe_environment_dict`.
This can take a while for large point sets, especially with high `N1`.

In [5]:
# Tune these before running
occlusion_lst = np.array([10, 25, 50, 75, 100, 125, 150, 200, 250, 300, 350, 400, 450, 500]) * 0.01
N1 = 1024  # Number of rays per point

if not MESH_PATH.exists():
    raise FileNotFoundError(f"Mesh file not found: {MESH_PATH}")

source_features = probe_environment_dict(
    named_points=source_points,
    mesh_path=str(MESH_PATH),
    occlusion_lst=occlusion_lst,
    direction_method="Fibonacci",
    N1=N1,
    N2=None,
    )

receiver_features = probe_environment_dict(
    named_points=receiver_points,
    mesh_path=str(MESH_PATH),
    occlusion_lst=occlusion_lst,
    direction_method="Fibonacci",
    N1=N1,
    N2=None,
    )

print(f"Source feature vectors: {len(source_features)}")
print(f"Receiver feature vectors: {len(receiver_features)}")
print("Feature length:", len(next(iter(source_features.values()))))

Probing points: 100%|██████████| 1000/1000 [02:31<00:00,  6.61it/s]


Source feature vectors: 20
Receiver feature vectors: 1000
Feature length: 6158


### Save Feature Tables
Persist source and receiver feature vectors for downstream fusion with spectrogram inputs.

In [12]:
# Convert dict outputs to DataFrames and persist
source_df = pd.DataFrame.from_dict(source_features, orient="index")
source_df.index.name = "point_name"

receiver_df = pd.DataFrame.from_dict(receiver_features, orient="index")
receiver_df.index.name = "point_name"

FEATURE_DIR = DATA_DIR / "features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

source_out = FEATURE_DIR / "source_features.parquet"
receiver_out = FEATURE_DIR / "receiver_features.parquet"

try:
    source_df.to_parquet(source_out)
    receiver_df.to_parquet(receiver_out)
    print(f"Saved source features: {source_out}")
    print(f"Saved receiver features: {receiver_out}")
except Exception as exc:
    # Fallback for pyarrow/pandas engine incompatibilities in some environments.
    source_csv = FEATURE_DIR / "source_features.csv"
    receiver_csv = FEATURE_DIR / "receiver_features.csv"
    source_df.to_csv(source_csv)
    receiver_df.to_csv(receiver_csv)
    print("Parquet write failed; saved CSV instead.")
    print(f"Reason: {type(exc).__name__}: {exc}")
    print(f"Saved source features: {source_csv}")
    print(f"Saved receiver features: {receiver_csv}")

print("source_df shape:", source_df.shape)
print("receiver_df shape:", receiver_df.shape)

Parquet write failed; saved CSV instead.
Reason: ArrowKeyError: A type extension with name pandas.period already defined
Saved source features: /Users/thejavanoob/Research/empty_room_data/exported_data/features/source_features.csv
Saved receiver features: /Users/thejavanoob/Research/empty_room_data/exported_data/features/receiver_features.csv
source_df shape: (20, 6158)
receiver_df shape: (1000, 6158)


### Build 1-D Pair Feature Vectors
Create pair-level static features (source + receiver) and keep each pair as a single 1-D conditioning vector.

In [19]:
def build_pair_feature_vector(row, source_features, receiver_features):
    s_idx = int(row["source_idx"])
    r_idx = int(row["receiver_idx"])
    s_name = f"source_{s_idx:02d}"
    r_name = f"source_{s_idx:02d}_rec_{r_idx:02d}"

    src_loc = np.asarray(row["source_position"], dtype=np.float32).reshape(-1)
    rec_loc = np.asarray(row["receiver_position"], dtype=np.float32).reshape(-1)
    src_feat = np.asarray(source_features[s_name], dtype=np.float32).reshape(-1)
    rec_feat = np.asarray(receiver_features[r_name], dtype=np.float32).reshape(-1)

    # Final conditioning order: [src_loc, rec_loc, src_features, rec_features]
    return np.concatenate([src_loc, rec_loc, src_feat, rec_feat], axis=0)

In [20]:
# Build 1-D pair feature vectors (no time-axis duplication)
pair_features_1d = {}

for row in all_metadata:
    s_idx = int(row["source_idx"])
    r_idx = int(row["receiver_idx"])
    pair_name = f"source{s_idx}_rec{r_idx}"
    pair_vec = build_pair_feature_vector(row, source_features, receiver_features)
    pair_features_1d[pair_name] = pair_vec

print("Pair feature vectors:", len(pair_features_1d))
print("Feature dimension per pair:", len(next(iter(pair_features_1d.values()))))

Pair feature vectors: 1000
Feature dimension per pair: 12322


In [21]:
# Save 1-D pair feature vectors
PAIR_FEATURE_DIR = FEATURE_DIR / "pair_features"
PAIR_FEATURE_DIR.mkdir(parents=True, exist_ok=True)

for pair_name, vec in pair_features_1d.items():
    np.save(PAIR_FEATURE_DIR / f"{pair_name}.npy", vec.astype(np.float32))

pair_df = pd.DataFrame(
    [{"pair_name": k, "feature_dim": int(v.shape[0])} for k, v in pair_features_1d.items()]
).sort_values("pair_name")
pair_df.to_csv(FEATURE_DIR / "pair_feature_dims.csv", index=False)

print(f"Saved 1-D pair features to: {PAIR_FEATURE_DIR}")
print(f"Saved feature dims to: {FEATURE_DIR / 'pair_feature_dims.csv'}")

Saved 1-D pair features to: /Users/thejavanoob/Research/empty_room_data/exported_data/features/pair_features
Saved feature dims to: /Users/thejavanoob/Research/empty_room_data/exported_data/features/pair_feature_dims.csv


### Create Train/Dev/Test Split for WAVs and Labels
Build deterministic splits using pair names shared by `.wav` files and 1-D label `.npy` files, then copy matched files into split folders and write CSV manifests.

In [22]:
import shutil

# Split configuration (must sum to 1.0)
TRAIN_RATIO = 0.8
DEV_RATIO = 0.1
TEST_RATIO = 0.1
SPLIT_SEED = 42
SPLIT_ROOT = DATA_DIR / "splits"

ratio_sum = TRAIN_RATIO + DEV_RATIO + TEST_RATIO
if not np.isclose(ratio_sum, 1.0):
    raise ValueError(f"Split ratios must sum to 1.0, got {ratio_sum}")

if not WAV_DIR.exists():
    raise FileNotFoundError(f"WAV directory not found: {WAV_DIR}")
if not PAIR_FEATURE_DIR.exists():
    raise FileNotFoundError(f"Pair label directory not found: {PAIR_FEATURE_DIR}")

wav_map = {p.stem: p for p in WAV_DIR.glob("*.wav")}
label_map = {p.stem: p for p in PAIR_FEATURE_DIR.glob("*.npy")}

shared_pairs = sorted(set(wav_map).intersection(label_map))
missing_wav = sorted(set(label_map) - set(wav_map))
missing_label = sorted(set(wav_map) - set(label_map))

if not shared_pairs:
    raise ValueError("No shared pair names between wav files and label files.")

rng = np.random.default_rng(SPLIT_SEED)
pairs = np.array(shared_pairs, dtype=object)
rng.shuffle(pairs)

n_total = len(pairs)
n_train = int(n_total * TRAIN_RATIO)
n_dev = int(n_total * DEV_RATIO)
n_test = n_total - n_train - n_dev

split_pairs = {
    "train": pairs[:n_train].tolist(),
    "dev": pairs[n_train:n_train + n_dev].tolist(),
    "test": pairs[n_train + n_dev:].tolist(),
}

for split_name in split_pairs:
    (SPLIT_ROOT / split_name / "wav").mkdir(parents=True, exist_ok=True)
    (SPLIT_ROOT / split_name / "labels").mkdir(parents=True, exist_ok=True)

rows = []
for split_name, names in split_pairs.items():
    for name in names:
        wav_src = wav_map[name]
        label_src = label_map[name]

        wav_dst = SPLIT_ROOT / split_name / "wav" / wav_src.name
        label_dst = SPLIT_ROOT / split_name / "labels" / label_src.name

        shutil.copy2(wav_src, wav_dst)
        shutil.copy2(label_src, label_dst)

        rows.append({
            "pair_name": name,
            "split": split_name,
            "wav_file": str(wav_dst.relative_to(DATA_DIR)),
            "label_file": str(label_dst.relative_to(DATA_DIR)),
        })

split_df = pd.DataFrame(rows).sort_values(["split", "pair_name"])
split_csv = SPLIT_ROOT / "split_manifest.csv"
split_df.to_csv(split_csv, index=False)

print(f"Split root: {SPLIT_ROOT}")
print(f"Total shared pairs: {n_total}")
print(f"Train/Dev/Test: {n_train}/{n_dev}/{n_test}")
print(f"Saved split manifest: {split_csv}")

if missing_wav:
    print(f"Labels without matching wav: {len(missing_wav)}")
if missing_label:
    print(f"Wavs without matching label: {len(missing_label)}")

Split root: /Users/thejavanoob/Research/empty_room_data/exported_data/splits
Total shared pairs: 1000
Train/Dev/Test: 800/100/100
Saved split manifest: /Users/thejavanoob/Research/empty_room_data/exported_data/splits/split_manifest.csv
